In [11]:
import pandas as pd
import numpy as np
from pathlib import Path

FILES = [
    "csv/beritamalang_media_articles.csv",
    "csv/kominfo_malangkab_articles.csv",
    "csv/malangposco_articles.csv",
    "csv/malangtimes_articles.csv",
    "csv/radar_malang_articles.csv",
    "csv/seputar_malang_articles.csv"
]

SOURCE_MAP = {
    "beritamalang_media_articles.csv": "beritamalang_media",
    "kominfo_malangkab_articles.csv": "kominfo_malangkab",
    "malangposco_articles.csv": "malangposco",
    "malangtimes_articles.csv": "malangtimes",
    "radar_malang_articles.csv": "radar_malang",
    "seputar_malang_articles.csv": "seputar_malang"
}

all_dfs = []

for file in FILES:

    df = pd.read_csv(file)

    df["source"] = SOURCE_MAP[
        Path(file).name
    ]

    for col in [
        "title",
        "url",
        "published_date",
        "author",
        "category",
        "content"
    ]:

        if col not in df.columns:
            df[col] = None

    df = df[
        [
            "source",
            "title",
            "url",
            "published_date",
            "author",
            "category",
            "content"
        ]
    ]

    all_dfs.append(df)

news = pd.concat(
    all_dfs,
    ignore_index=True
)

news = news.replace(
    {np.nan: None}
)

# =====================================================
# NORMALISASI CATEGORY
# =====================================================

VALID_CATEGORY = {
    "pendidikan": "Pendidikan",
    "kesehatan": "Kesehatan",
    "ekonomi": "Ekonomi",
    "sosial": "Sosial"
}

def normalize_category(x):

    if pd.isna(x) or x is None:
        return None

    x = str(x).lower()

    for key, value in VALID_CATEGORY.items():

        if key in x:
            return value

    return None

news["category"] = news["category"].apply(
    normalize_category
)

# =====================================================
# NORMALISASI TANGGAL
# =====================================================

def parse_date(x):

    if pd.isna(x) or x is None:
        return pd.NaT

    x = str(x).strip()

    formats = [

        "%Y-%m-%d %H:%M:%S",

        "%Y-%m-%dT%H:%M:%S%z",

        "%B %d, %Y",

        "%d %B %y, %H:%M",

        "%d %B %Y",

        "%d - %b - %Y, %H:%M"
    ]

    for fmt in formats:

        try:
            return pd.to_datetime(
                x,
                format=fmt
            )
        except:
            pass

    return pd.to_datetime(
        x,
        errors="coerce"
    )

news["published_date"] = (
    news["published_date"]
    .apply(parse_date)
)

news["published_date"] = pd.to_datetime(
    news["published_date"],
    utc=True,
    errors="coerce"
)

# =====================================================
# FILTER 4 BULAN
# =====================================================

cutoff = (
    pd.Timestamp.now(tz="UTC")
    - pd.DateOffset(months=4)
)

news = news[
    news["published_date"] >= cutoff
]

# =====================================================
# DEDUP
# =====================================================

news = (
    news
    .drop_duplicates(
        subset=["url"]
    )
    .reset_index(drop=True)
)

# =====================================================
# CEK HASIL
# =====================================================

print(news.shape)

print("\nSOURCE COUNT")
print(
    news["source"]
    .value_counts()
)

print("\nCATEGORY COUNT")
print(
    news["category"]
    .value_counts(
        dropna=False
    )
)

print("\nMISSING CONTENT")
print(
    news["content"]
    .isna()
    .sum()
)

# =====================================================
# SAVE
# =====================================================

news.to_csv(
    "csv/all_news.csv",
    index=False
)

print(
    f"\nsaved {len(news)} rows"
)

(108, 7)

SOURCE COUNT
source
beritamalang_media    30
radar_malang          30
malangtimes           27
malangposco           18
seputar_malang         3
Name: count, dtype: int64

CATEGORY COUNT
category
NaN           78
Pendidikan    30
Name: count, dtype: int64

MISSING CONTENT
0

saved 108 rows
